# Word-Level Duration Poisson GLM Orchestrator

This notebook now acts as a lightweight orchestrator instead of fitting every patient in-kernel.

It:

1. discovers per-patient inputs under `vad_new/{patient}`
2. shows which patients are ready vs. missing inputs
3. submits one SLURM job per patient
4. loads only saved per-patient result summaries back into the notebook

Heavy modeling logic lives in [poisson_glm_worker.py](/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_encoding_analysis/poisson_glm_worker.py).

Preferred input layout:

- `vad_new/{patient}/embeddings/{patient}_gpt2_embeddings.npy`
- `vad_new/{patient}/neural_embeddings/word_spike_counts_offsets_all.npy`
- `vad_new/{patient}/neural_embeddings/word_durs.npy`

Legacy fallback under `vad_new/{patient}/all_convo_recording/` is still supported by the worker.


In [1]:
from pathlib import Path
import subprocess

import dill as pickle
import pandas as pd


In [2]:
# --------------------
# User-facing settings
# --------------------
PROJ_DIR = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247')
VAD_ROOT = PROJ_DIR / 'vad_new'
WORKER_PATH = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_encoding_analysis/poisson_glm_worker.py')
PYTHON_BIN = '/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python3'

PATIENTS = None      # None => scan all Y* patient folders under vad_new
FORCE_RESUBMIT = False

# Model settings
SPIKE_OFFSET_IDX = 0  # counts array has shape (1, words, neurons) — only offset 0 exists
GPT2_LAYER = -1
N_PCA = 100
OUTER_SPLITS = 5
INNER_SPLITS = 5
N_ALPHAS = 30
ALPHA_LOW = -3.0
ALPHA_HIGH = 3.0
N_SHUFFLES = 50

# Compute mode — set False to submit CPU-only jobs (no GPU slot required)
USE_GPU = True

# SLURM settings
CONDA_ACTIVATE = 'source /scratch/tahaismail424/.bash_profile && conda activate speech_247_env'
if USE_GPU:
    PARTITION = 'Universal'
    GRES      = 'gpu:1'
    CPUS      = 8
    MEM       = '64G'
    TIME      = '24:00:00'
else:
    PARTITION = 'Universal'
    GRES      = ''       # no GPU
    CPUS      = 16
    MEM       = '32G'
    TIME      = '48:00:00'

RUN_NAME = 'word_level_duration_cv_all_n'
GLOBAL_LOGS_DIR = VAD_ROOT / f'{RUN_NAME}_slurm_logs'
GLOBAL_SCRIPTS_DIR = VAD_ROOT / f'{RUN_NAME}_slurm_scripts'
GLOBAL_LOGS_DIR.mkdir(parents=True, exist_ok=True)
GLOBAL_SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

print('vad root:   ', VAD_ROOT)
print('worker:     ', WORKER_PATH)
print('logs dir:   ', GLOBAL_LOGS_DIR)
print('scripts dir:', GLOBAL_SCRIPTS_DIR)
print(f'compute:     {"GPU" if USE_GPU else "CPU"} | partition={PARTITION}')

vad root:    /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new
worker:      /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_encoding_analysis/poisson_glm_worker.py
logs dir:    /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/word_level_duration_cv_all_n_slurm_logs
scripts dir: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/word_level_duration_cv_all_n_slurm_scripts
compute:     GPU | partition=Universal


In [3]:
def first_existing(paths):
    for path in paths:
        if path is not None and Path(path).exists():
            return Path(path)
    return None


def resolve_patient_paths(patient: str) -> dict:
    patient_root = VAD_ROOT / patient
    embeddings_path = first_existing([
        patient_root / 'embeddings' / f'{patient}_gpt2_embeddings.npy',
        patient_root / 'all_convo_recording' / 'all_words_filtered_all_layers_gpt2.npy',
    ])
    counts_path = first_existing([
        patient_root / 'neural_embeddings' / 'word_spike_counts_offsets_all.npy',
        patient_root / 'all_convo_recording' / 'word_spike_counts_offsets_all.npy',
    ])
    durations_path = first_existing([
        patient_root / 'neural_embeddings' / 'word_durs.npy',
        patient_root / 'all_convo_recording' / 'word_durs.npy',
    ])

    out_dir = patient_root / 'encoding' / RUN_NAME
    result_pkl = out_dir / f'{patient}_encoding_results_cv.pkl'
    result_tar = out_dir / f'{patient}_encoding_models_cv.tar'
    meta_json = out_dir / f'{patient}_meta.json'
    success_path = out_dir / f'{patient}_SUCCESS'
    error_path = out_dir / f'{patient}_error.txt'

    return {
        'patient': patient,
        'patient_root': patient_root,
        'embeddings_path': embeddings_path,
        'counts_path': counts_path,
        'durations_path': durations_path,
        'out_dir': out_dir,
        'result_pkl': result_pkl,
        'result_tar': result_tar,
        'meta_json': meta_json,
        'success_path': success_path,
        'error_path': error_path,
    }


def build_status_df(patients=None) -> pd.DataFrame:
    if patients is None:
        patients = sorted([p.name for p in VAD_ROOT.iterdir() if p.is_dir() and p.name.startswith('Y')])
    rows = []
    for patient in patients:
        info = resolve_patient_paths(patient)
        row = {
            **info,
            'has_embeddings': info['embeddings_path'] is not None,
            'has_counts': info['counts_path'] is not None,
            'has_durations': info['durations_path'] is not None,
            'has_success': info['success_path'].exists(),
            'has_error': info['error_path'].exists(),
            'has_result_pkl': info['result_pkl'].exists(),
            'has_result_tar': info['result_tar'].exists(),
        }
        row['ready_inputs'] = row['has_embeddings'] and row['has_counts'] and row['has_durations']
        row['needs_submit'] = row['ready_inputs'] and (FORCE_RESUBMIT or not row['has_success'])
        rows.append(row)
    return pd.DataFrame(rows).sort_values('patient').reset_index(drop=True)


In [4]:
status_df = build_status_df(PATIENTS)
print('patients:', len(status_df))
print('ready inputs:', int(status_df['ready_inputs'].sum()))
print('completed:', int(status_df['has_success'].sum()))
print('errors:', int(status_df['has_error'].sum()))

status_df[[
    'patient', 'has_embeddings', 'has_counts', 'has_durations',
    'ready_inputs', 'has_success', 'has_error', 'needs_submit'
]]


patients: 21
ready inputs: 21
completed: 21
errors: 2


,patient,has_embeddings,has_counts,has_durations,ready_inputs,has_success,has_error,needs_submit
0,YEY,True,True,True,True,True,False,False
1,YEZ,True,True,True,True,True,False,False
2,YFA,True,True,True,True,True,False,False
3,YFB,True,True,True,True,True,False,False
4,YFC,True,True,True,True,True,False,False
5,YFD,True,True,True,True,True,False,False
6,YFE,True,True,True,True,True,False,False
7,YFF,True,True,True,True,True,False,False
8,YFG,True,True,True,True,True,False,False
9,YFI,True,True,True,True,True,False,False


In [5]:
submitted = []
failed = []

for _, row in status_df.iterrows():
    patient = row['patient']
    if not row['needs_submit']:
        continue

    info = resolve_patient_paths(patient)
    info['out_dir'].mkdir(parents=True, exist_ok=True)
    reset_line = f"rm -f {info['success_path']} {info['error_path']}" if FORCE_RESUBMIT else 'true'

    gres_lines = [f'#SBATCH --gres={GRES}'] if GRES else []
    lines = [
        '#!/bin/bash',
        f'#SBATCH --job-name=glm_{patient}',
        f'#SBATCH --partition={PARTITION}',
        *gres_lines,
        f'#SBATCH --cpus-per-task={CPUS}',
        f'#SBATCH --qos=default_tier',
        f'#SBATCH --exclude=guppy-1',
        f'#SBATCH --mem={MEM}',
        f'#SBATCH --time={TIME}',
        f'#SBATCH --output={GLOBAL_LOGS_DIR}/{patient}_%j.out',
        f'#SBATCH --error={GLOBAL_LOGS_DIR}/{patient}_%j.err',
        '',
        'set -eo pipefail',
        '',
        CONDA_ACTIVATE,
        '',
        'echo "HOSTNAME: $(hostname)"',
        'echo "START: $(date)"',
        f'echo "PATIENT: {patient}"',
        '',
        reset_line,
        '',
        f'{PYTHON_BIN} {WORKER_PATH} {patient} {VAD_ROOT} {info["out_dir"]} ' +
        f'--spike-offset-idx {SPIKE_OFFSET_IDX} ' +
        f'--gpt2-layer {GPT2_LAYER} ' +
        f'--n-pca {N_PCA} ' +
        f'--outer-splits {OUTER_SPLITS} ' +
        f'--inner-splits {INNER_SPLITS} ' +
        f'--n-alphas {N_ALPHAS} ' +
        f'--alpha-low {ALPHA_LOW} ' +
        f'--alpha-high {ALPHA_HIGH} ' +
        f'--n-shuffles {N_SHUFFLES} ' +
        f'--embeddings-path {info["embeddings_path"]} ' +
        f'--counts-path {info["counts_path"]} ' +
        f'--durations-path {info["durations_path"]}',
        '',
        'echo "END: $(date)"',
    ]
    sbatch_text = '\n'.join(lines)

    sbatch_path = GLOBAL_SCRIPTS_DIR / f'{patient}.sbatch'
    sbatch_path.write_text(sbatch_text)

    try:
        res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
        submitted.append((patient, res.stdout.strip()))
        print(f'submitted: {patient} -> {res.stdout.strip()}')
    except subprocess.CalledProcessError as exc:
        failed.append((patient, exc.stderr.strip()))
        print(f'FAILED SUBMISSION: {patient}')
        print(exc.stderr)

print(f'submitted={len(submitted)}, failed={len(failed)}')

submitted=0, failed=0


In [ ]:
status_df = build_status_df(PATIENTS)
status_df[['patient', 'ready_inputs', 'has_success', 'has_error', 'has_result_pkl', 'has_result_tar']]


In [ ]:
error_rows = status_df[status_df['has_error']].copy()
print(f'patients with error files: {len(error_rows)}')
for _, row in error_rows.head(5).iterrows():
    print('=' * 100)
    print(row['patient'])
    print(row['error_path'])
    print(row['error_path'].read_text()[:4000])


In [ ]:
records = []
loaded = []

for _, row in status_df.iterrows():
    if not row['has_result_pkl']:
        continue
    with open(row['result_pkl'], 'rb') as f:
        df = pickle.load(f)
    summary_df = df[df['is_summary'] == True].copy() if 'is_summary' in df.columns else pd.DataFrame()
    if summary_df.empty:
        continue
    summary_df['patient'] = row['patient']
    records.append(summary_df)
    loaded.append(row['patient'])

if records:
    all_summary = pd.concat(records, ignore_index=True)
    print('loaded patients:', loaded)
    print('summary rows:', len(all_summary))
    display(all_summary.head())

    patient_level = all_summary.groupby('patient').agg(
        n_neurons=('neuron_idx', 'nunique'),
        pseudo_r2_mean=('pseudo_r2_mean', 'mean'),
        pearson_corr_mean=('pearson_corr_mean', 'mean'),
        spearman_corr_mean=('spearman_corr_mean', 'mean'),
    ).reset_index()
    display(patient_level.sort_values('patient'))
else:
    print('No completed patient result pickles found yet.')
